### 🧪 ContextBuilderProvider Log Confirmation — Minimal EDA View

This cell loads and expands `ContextOutputSchema` entries from the provider logs to validate that context logs are being written correctly. It flattens the nested `context` structure for the current `LintingContextProvider`, extracting:

- Top-level metadata (`file_path`, `score_name`, `score_value`, `log_count`)
- Linting score components (e.g. `ruff_score`, `black_score`, etc.)
- Basic latency and timestamp metadata

While these features support lightweight EDA, the real downstream use case is to treat the full context (including source code and conversation log) as **textual input** for LLM-based agents. Future analysis will likely involve:

- Combining `summary` and `context` into a unified narrative block
- Embedding or tokenizing the resulting text for semantic analysis
- Comparing output quality across systems based on context payloads, not just raw metrics

For now, this cell serves as a **sanity check that structured logging is consistent and complete**.


In [12]:
# 🧪 ContextBuilderProvider Log Expansion — Featurized Context Output

import os, sys, json
import pandas as pd
from pathlib import Path

# 📁 Set up paths
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

LOG_PATH = Path("tests/backup/provider_logs.csv")
assert LOG_PATH.exists(), "❌ provider_logs.csv not found"

# 📥 Load log data
df = pd.read_csv(LOG_PATH)

# 🔍 Filter only CONTEXT provider logs
df_context = df[df["provider_type"] == "PROVIDER_TYPE.CONTEXT"].copy()

# 🧼 Safe JSON parse
def safe_json_parse(val):
    try:
        return json.loads(val) if isinstance(val, str) else {}
    except Exception:
        return {}

df_context["output_parsed"] = df_context["output"].apply(safe_json_parse)

# 🧩 Extract ContextOutputSchema fields only
df_context["summary"] = df_context["output_parsed"].apply(lambda x: x.get("summary"))
df_context["context"] = df_context["output_parsed"].apply(lambda x: x.get("context"))

# 🧬 Expand context structure for featurization
df_context["file_path"] = df_context["context"].apply(lambda x: x.get("file_path"))
df_context["source_code"] = df_context["context"].apply(lambda x: x.get("source_code"))
df_context["score_name"] = df_context["context"].apply(lambda x: x.get("score", {}).get("name"))
df_context["score_value"] = df_context["context"].apply(lambda x: x.get("score", {}).get("value"))
df_context["log_count"] = df_context["context"].apply(lambda x: len(x.get("conversation_log", [])))

components = [
    "ruff_score", "ruff_violations", "black_score", "mypy_score",
    "tool_failure_count", "weight_ruff", "weight_black", "weight_mypy"
]

for c in components:
    df_context[c] = df_context["context"].apply(lambda x: x.get("score", {}).get("components", {}).get(c))

# 📐 Select final feature columns
feature_cols = [
    "timestamp", "file_path", "score_name", "score_value", "log_count",
    "ruff_score", "ruff_violations", "black_score", "mypy_score",
    "tool_failure_count", "weight_ruff", "weight_black", "weight_mypy",
    "latency_ms", "summary"
]

df_features_view = df_context[feature_cols].sort_values("timestamp")

pd.set_option("display.max_colwidth", 200)
df_features_view.sort_values("timestamp").head(20)


,timestamp,file_path,score_name,score_value,log_count,ruff_score,ruff_violations,black_score,mypy_score,tool_failure_count,weight_ruff,weight_black,weight_mypy,latency_ms,summary
4,2025-06-16 01:51:50.425306+00:00,C:\Repos\codecritic\working_files\bad_indentation..__state_2051504163.py,linting_score,0.90,0,0.9,1.0,0.9,0.9,0.0,0.7,0.2,0.1,5136,"Context for bad_indentation..__state_2051504163.py, 0.9 score, 0 log entries"
77,2025-06-16 01:52:08.489401+00:00,C:\Repos\codecritic\working_files\complex_function..__state_2052084842.py,linting_score,0.89,0,0.9,1.0,0.9,0.8,0.0,0.7,0.2,0.1,683,"Context for complex_function..__state_2052084842.py, 0.89 score, 0 log entries"
150,2025-06-16 01:52:23.051525+00:00,C:\Repos\codecritic\working_files\long_line..__state_2052230464.py,linting_score,0.95,0,1.0,0.0,0.9,0.7,0.0,0.7,0.2,0.1,683,"Context for long_line..__state_2052230464.py, 0.95 score, 0 log entries"
223,2025-06-16 01:52:36.428811+00:00,C:\Repos\codecritic\working_files\mixed_tabs_spaces..__state_2052364237.py,linting_score,0.96,0,1.0,0.0,0.9,0.8,0.0,0.7,0.2,0.1,655,"Context for mixed_tabs_spaces..__state_2052364237.py, 0.96 score, 0 log entries"
296,2025-06-16 01:52:49.937272+00:00,C:\Repos\codecritic\working_files\no_doc_type..__state_2052499292.py,linting_score,0.96,0,1.0,0.0,0.9,0.8,0.0,0.7,0.2,0.1,691,"Context for no_doc_type..__state_2052499292.py, 0.96 score, 0 log entries"
369,2025-06-16 01:53:04.066558+00:00,C:\Repos\codecritic\working_files\off_by_one..__state_2053040595.py,linting_score,0.98,0,1.0,0.0,1.0,0.8,0.0,0.7,0.2,0.1,687,"Context for off_by_one..__state_2053040595.py, 0.98 score, 0 log entries"
442,2025-06-16 01:53:17.592227+00:00,C:\Repos\codecritic\working_files\shadow_builtin..__state_2053175862.py,linting_score,1.00,0,1.0,0.0,1.0,1.0,0.0,0.7,0.2,0.1,664,"Context for shadow_builtin..__state_2053175862.py, 1.0 score, 0 log entries"
515,2025-06-16 01:53:30.766264+00:00,C:\Repos\codecritic\working_files\unused_import..__state_2053307622.py,linting_score,0.91,0,0.9,1.0,0.9,1.0,0.0,0.7,0.2,0.1,681,"Context for unused_import..__state_2053307622.py, 0.91 score, 0 log entries"
588,2025-06-16 01:53:44.554244+00:00,C:\Repos\codecritic\working_files\valid_simple..__state_2053445480.py,linting_score,0.98,0,1.0,0.0,1.0,0.8,0.0,0.7,0.2,0.1,676,"Context for valid_simple..__state_2053445480.py, 0.98 score, 0 log entries"
661,2025-06-16 01:53:57.313943+00:00,C:\Repos\codecritic\working_files\wrong_type_hint..__state_2053573079.py,linting_score,0.97,0,1.0,0.0,1.0,0.7,0.0,0.7,0.2,0.1,694,"Context for wrong_type_hint..__state_2053573079.py, 0.97 score, 0 log entries"
